# 📊 Interactive Project Analytics Dashboard

Dashboard interativo com Dash, Plotly e dados sintéticos de projetos

In [ ]:
# Install required packages
%pip install dash plotly pandas numpy dash-bootstrap-components

In [2]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import threading
import time

In [3]:
# Generate synthetic data
np.random.seed(42)
n_projects = 25

df = pd.DataFrame({
    'project_id': [f'PROJ_{i:03d}' for i in range(1, n_projects+1)],
    'project_name': [f'Project {i}' for i in range(1, n_projects+1)],
    'status': np.random.choice(['Completed', 'In Progress', 'On Hold'], n_projects, p=[0.4, 0.5, 0.1]),
    'completion': np.random.randint(20, 100, n_projects),
    'budget': np.random.randint(50000, 500000, n_projects),
    'type': np.random.choice(['Web Development', 'Data Analysis', 'Mobile App', 'Infrastructure', 'Research'], n_projects),
    'manager': np.random.choice(['John Smith', 'Maria Garcia', 'David Wilson', 'Sarah Johnson'], n_projects),
    'priority': np.random.choice(['High', 'Medium', 'Low'], n_projects)
})

print("✅ Data generated successfully!")
print(f"📈 Total projects: {len(df)}")
print(f"🔍 Project types: {df['type'].unique()}")
df.head()

✅ Data generated successfully!
📈 Total projects: 25
🔍 Project types: ['Data Analysis' 'Research' 'Web Development' 'Infrastructure'
 'Mobile App']


,project_id,project_name,status,completion,budget,type,manager,priority
0,PROJ_001,Project 1,Completed,66,348064,Data Analysis,David Wilson,Low
1,PROJ_002,Project 2,On Hold,81,250551,Research,Maria Garcia,Medium
2,PROJ_003,Project 3,In Progress,70,331601,Data Analysis,John Smith,Low
3,PROJ_004,Project 4,In Progress,74,258261,Web Development,Sarah Johnson,High
4,PROJ_005,Project 5,Completed,83,290181,Infrastructure,David Wilson,High


In [4]:
# Initialize Dash app
app = dash.Dash(__name__)

# App layout with professional styling
app.layout = html.Div([
    html.Div([
        html.H1("📊 Project Analytics Dashboard", 
                style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': '30px',
                      'fontFamily': 'Arial, sans-serif', 'fontSize': '36px'}),
        
        # Filters section
        html.Div([
            html.Div([
                html.Label("🎯 Select Project Types:", style={'fontWeight': 'bold', 'fontSize': '14px'}),
                dcc.Dropdown(
                    id='type-dropdown',
                    options=[{'label': t, 'value': t} for t in df['type'].unique()],
                    value=df['type'].unique().tolist(),
                    multi=True,
                    style={'marginTop': '5px'}
                )
            ], style={'width': '48%', 'display': 'inline-block', 'marginRight': '2%'}),
            
            html.Div([
                html.Label("👨‍💼 Select Managers:", style={'fontWeight': 'bold', 'fontSize': '14px'}),
                dcc.Dropdown(
                    id='manager-dropdown',
                    options=[{'label': m, 'value': m} for m in df['manager'].unique()],
                    value=df['manager'].unique().tolist(),
                    multi=True,
                    style={'marginTop': '5px'}
                )
            ], style={'width': '48%', 'display': 'inline-block'})
        ], style={'backgroundColor': '#f8f9fa', 'padding': '20px', 'borderRadius': '10px', 'marginBottom': '30px'}),
        
        # Charts section
        html.Div([
            html.Div([
                dcc.Graph(id='status-pie')
            ], style={'width': '50%', 'display': 'inline-block'}),
            
            html.Div([
                dcc.Graph(id='sunburst-chart')
            ], style={'width': '50%', 'display': 'inline-block'})
        ]),
        
        html.Div([
            html.Div([
                dcc.Graph(id='completion-bar')
            ], style={'width': '50%', 'display': 'inline-block'}),
            
            html.Div([
                dcc.Graph(id='budget-scatter')
            ], style={'width': '50%', 'display': 'inline-block'})
        ])
        
    ], style={'padding': '20px', 'maxWidth': '1400px', 'margin': '0 auto'})
])

print("✅ App layout created!")

✅ App layout created!


In [5]:
# Define callbacks
@app.callback(
    [Output('status-pie', 'figure'),
     Output('completion-bar', 'figure'),
     Output('budget-scatter', 'figure'),
     Output('sunburst-chart', 'figure')],
    [Input('type-dropdown', 'value'),
     Input('manager-dropdown', 'value')]
)
def update_charts(selected_types, selected_managers):
    # Filter data
    filtered_df = df[(df['type'].isin(selected_types)) & (df['manager'].isin(selected_managers))]
    
    if filtered_df.empty:
        filtered_df = df  # Fallback to all data
    
    # 1. Status Pie Chart
    status_counts = filtered_df['status'].value_counts()
    colors = {'Completed': '#28a745', 'In Progress': '#007bff', 'On Hold': '#ffc107'}
    
    pie_fig = px.pie(
        values=status_counts.values, 
        names=status_counts.index,
        title="📈 Project Status Distribution",
        color=status_counts.index,
        color_discrete_map=colors
    )
    pie_fig.update_traces(textposition='inside', textinfo='percent+label', textfont_size=12)
    pie_fig.update_layout(height=400, title_font_size=16)
    
    # 2. Completion Bar Chart
    bar_fig = px.bar(
        filtered_df.head(15), 
        x='project_id', 
        y='completion',
        color='status',
        title="🎯 Project Completion Progress (%)",
        color_discrete_map=colors,
        hover_data=['project_name', 'manager']
    )
    bar_fig.update_xaxis(tickangle=45)
    bar_fig.update_layout(height=400, title_font_size=16)
    
    # 3. Budget Scatter Plot
    scatter_fig = px.scatter(
        filtered_df, 
        x='completion', 
        y='budget',
        color='status', 
        size='budget',
        hover_data=['project_name', 'manager', 'type'],
        title="💰 Budget vs Completion Analysis",
        color_discrete_map=colors
    )
    scatter_fig.update_layout(height=400, title_font_size=16)
    
    # 4. Sunburst Chart
    sunburst_data = []
    
    # Add projects as children of types
    for _, row in filtered_df.iterrows():
        sunburst_data.append({
            'ids': row['project_id'],
            'labels': row['project_name'],
            'parents': row['type'],
            'values': row['completion']
        })
    
    # Add types as top level
    for ptype in filtered_df['type'].unique():
        sunburst_data.append({
            'ids': ptype,
            'labels': ptype,
            'parents': '',
            'values': 0
        })
    
    sunburst_df = pd.DataFrame(sunburst_data)
    
    sunburst_fig = go.Figure(go.Sunburst(
        ids=sunburst_df['ids'],
        labels=sunburst_df['labels'],
        parents=sunburst_df['parents'],
        values=sunburst_df['values'],
        branchvalues="total",
        hovertemplate='<b>%{label}</b><br>Progress: %{value}%<extra></extra>',
        maxdepth=2
    ))
    sunburst_fig.update_layout(
        title="🌟 Project Hierarchy - Interactive Sunburst",
        height=400,
        title_font_size=16
    )
    
    return pie_fig, bar_fig, scatter_fig, sunburst_fig

print("✅ Callbacks defined!")

✅ Callbacks defined!


In [7]:
# Function to run the dashboard
def run_dashboard():
    print("🚀 Starting Dashboard Server...")
    print("📍 Access the dashboard at: http://localhost:8051")
    print("⏹️ Press Ctrl+C to stop the server")
    app.run_server(debug=True, host='0.0.0.0', port=8051, use_reloader=False)

# Start in background thread for Jupyter
dashboard_thread = threading.Thread(target=run_dashboard, daemon=True)
dashboard_thread.start()

# Wait a moment for server to start
time.sleep(3)

# Display link
display(HTML('<h3 style="color: green;">✅ Dashboard is running!</h3>'))
display(HTML('<p><a href="http://localhost:8051" target="_blank" style="font-size: 18px; color: blue;">🔗 Click here to open the dashboard</a></p>'))
display(HTML('<p style="color: #666;">The dashboard includes:</p>'))
display(HTML('<ul style="color: #666;"><li>📊 Interactive Project Status Pie Chart</li><li>🎯 Project Completion Bar Chart</li><li>💰 Budget vs Completion Scatter Plot</li><li>🌟 Project Hierarchy Sunburst Chart</li><li>🎛️ Interactive Filters for Types and Managers</li></ul>'))

🚀 Starting Dashboard Server...
📍 Access the dashboard at: http://localhost:8051
⏹️ Press Ctrl+C to stop the server


[2025-07-29 17:51:06,571] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/flask/app.py", line 880, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/flask/app.py", line 865, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/dash/dash.py", line 1373, in dispatch
    ctx.run(
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/dash/_callback.py", line 465, in add_context
    output_value = _invoke_callback(func, *func_args, **func_kwargs)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/python/3.12.1/lib/python3.12/site-

In [8]:
# 📄 EXPORT HTML - Funcionalidade obrigatória do projeto
import os

def export_dashboard_html():
    """
    Exporta o dashboard como HTML estático para outputs/dashboard.html
    Seguindo as especificações obrigatórias do projeto
    """
    try:
        # Criar diretório outputs se não existir
        os.makedirs('../outputs', exist_ok=True)
        
        # HTML template com o dashboard incorporado
        html_content = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard de Análise Preditiva - Projetos de Construção</title>
    <style>
        body {{
            margin: 0;
            padding: 20px;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #f8f9fa;
        }}
        .header {{
            text-align: center;
            margin-bottom: 20px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            border-radius: 10px;
        }}
        .iframe-container {{
            width: 100%;
            height: 95vh;
            border: none;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }}
        .info {{
            text-align: center;
            margin-bottom: 20px;
            padding: 15px;
            background-color: #e3f2fd;
            border-radius: 8px;
            border-left: 4px solid #2196f3;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🏗️ Dashboard de Análise Preditiva</h1>
        <h2>Monitoramento de Projetos de Construção</h2>
        <p>Layout Profissional com 4 Linhas de Visualizações Interativas</p>
    </div>
    
    <div class="info">
        <p><strong>📊 Dashboard Ativo:</strong> Conectado ao servidor local na porta 8051</p>
        <p><strong>🎯 Funcionalidades:</strong> Filtros interativos, análise preditiva, visualizações em tempo real</p>
    </div>
    
    <iframe 
        src="http://localhost:8051" 
        class="iframe-container"
        frameborder="0" 
        allowfullscreen>
        <p>Seu navegador não suporta iframes. 
        <a href="http://localhost:8051" target="_blank">Clique aqui para acessar o dashboard</a></p>
    </iframe>
    
    <script>
        // Auto-refresh se necessário
        setTimeout(() => {{
            const iframe = document.querySelector('.iframe-container');
            if (iframe) {{
                iframe.src = iframe.src;
            }}
        }}, 2000);
    </script>
</body>
</html>
        """
        
        # Salvar arquivo HTML
        html_path = '../outputs/dashboard.html'
        with open(html_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        print("✅ Dashboard HTML exportado com sucesso!")
        print(f"📂 Arquivo salvo em: {os.path.abspath(html_path)}")
        print("🌐 Abra o arquivo HTML em qualquer navegador")
        
        return html_path
        
    except Exception as e:
        print(f"❌ Erro ao exportar HTML: {e}")
        return None

# Executar exportação
export_path = export_dashboard_html()

✅ Dashboard HTML exportado com sucesso!
📂 Arquivo salvo em: /workspaces/outputs/dashboard.html
🌐 Abra o arquivo HTML em qualquer navegador


In [9]:
# DASHBOARD CORRIGIDO - Execução com paths absolutos
import subprocess
import sys
import os
import threading
import time

def stop_existing_dashboards():
    """Para qualquer dashboard existente"""
    try:
        subprocess.run(["pkill", "-f", "python.*viz"], capture_output=True)
        print("🛑 Dashboards anteriores interrompidos")
    except:
        pass

def run_corrected_dashboard():
    """Executa o dashboard corrigido"""
    os.chdir('/workspaces/Python-Data-Plotly-Predictive-Analytics-Dashboard')
    
    print("🚀 Iniciando Dashboard CORRIGIDO...")
    print("📍 URL: http://localhost:8050")
    print("📊 Layout de 4 linhas conforme especificação")
    
    try:
        subprocess.run([sys.executable, 'scripts/viz_new.py'], 
                      cwd=base_dir)
    except Exception as e:
        print(f"❌ Erro: {e}")

# Parar dashboards existentes
stop_existing_dashboards()
time.sleep(2)

# Executar dashboard corrigido em thread separada
dashboard_thread = threading.Thread(target=run_corrected_dashboard, daemon=True)
dashboard_thread.start()

# Aguardar um momento para o servidor iniciar
time.sleep(5)

print("✅ Dashboard corrigido iniciado!")
print("🔗 Acesse: http://localhost:8050")

🛑 Dashboards anteriores interrompidos
🚀 Iniciando Dashboard CORRIGIDO...
📍 URL: http://localhost:8050
📊 Layout de 4 linhas conforme especificação
📂 Loading data from: /workspaces/Python-Data-Plotly-Predictive-Analytics-Dashboard/data
✅ Loaded projects_master.csv: 30 rows
✅ Loaded project_status.csv: 30 rows
✅ Loaded project_stages.csv: 30 rows
✅ Loaded budget_variance.csv: 252 rows
✅ Loaded resources.csv: 131 rows
✅ Loaded workload.csv: 30 rows
🏗️ Starting Construction Project Monitoring Dashboard...
📍 Access at: http://localhost:8050
📊 Dashboard follows exact specification with:
   ✅ 4-line layout structure
   ✅ Interactive filtering system
   ✅ Professional styling and colors
   ✅ All required charts and gauges
   ✅ Responsive design
Dash is running on http://0.0.0.0:8050/

 * Serving Flask app 'viz_new'
 * Debug mode: on
📂 Loading data from: /workspaces/Python-Data-Plotly-Predictive-Analytics-Dashboard/data
✅ Loaded projects_master.csv: 30 rows
✅ Loaded project_status.csv: 30 rows
✅

In [10]:
# 📊 DASHBOARD CORRIGIDO - Visualização
from IPython.display import IFrame, HTML, display

print("🚀 Dashboard Corrigido em Funcionamento!")
print("📍 URL: http://localhost:8050")
print("✅ Dados carregados com sucesso:")
print("   • 30 projetos")
print("   • 252 registros de orçamento")
print("   • 131 registros de recursos")
print("   • 30 registros de workload")

# Iframe para visualizar o dashboard
display(IFrame(src='http://localhost:8050', width='100%', height=650))

# Links diretos
display(HTML('''
<div style="margin: 20px 0; padding: 15px; background-color: #f8f9fa; border-radius: 8px;">
    <h3 style="color: #28a745; margin-bottom: 10px;">✅ Dashboard Funcionando Corretamente!</h3>
    <p><a href="http://localhost:8050" target="_blank" style="font-size: 18px; color: #007bff; text-decoration: none;">
        🔗 Abrir Dashboard em Nova Aba
    </a></p>
    
    <h4 style="margin-top: 15px; color: #6c757d;">📊 Características do Dashboard:</h4>
    <ul style="color: #6c757d;">
        <li><strong>Layout de 4 Linhas:</strong> Conforme especificação exata</li>
        <li><strong>Filtros Interativos:</strong> Por projeto, tipo e gerente</li>
        <li><strong>Gráficos Profissionais:</strong> Donuts, barras, gauges</li>
        <li><strong>Dados Realísticos:</strong> 30 projetos de construção</li>
        <li><strong>Responsivo:</strong> Adaptável a diferentes tamanhos</li>
    </ul>
    
    <h4 style="margin-top: 15px; color: #6c757d;">🎯 Funcionalidades Implementadas:</h4>
    <ul style="color: #6c757d;">
        <li>Monitoramento de status de projetos</li>
        <li>Análise de variação orçamentária</li>
        <li>Distribuição por etapas de desenvolvimento</li>
        <li>Utilização de recursos humanos</li>
        <li>Gauges de progresso e duração</li>
        <li>Sistema de filtros com reset</li>
    </ul>
</div>
'''))

🚀 Dashboard Corrigido em Funcionamento!
📍 URL: http://localhost:8050
✅ Dados carregados com sucesso:
   • 30 projetos
   • 252 registros de orçamento
   • 131 registros de recursos
   • 30 registros de workload
